# Programación declarativa @ GIA - URJC
## Curso 25-26
## Convocatoria ordinaria
## Prueba 2

La duración de la prueba es de 1 hora y 30 minutos.


# Preámbulo

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

In [ ]:
object Signatures:
    abstract class List[A]:
        
        // Common HOFs
        def foldRight[B](nil: B)(cons: (A, B) => B): B
        def foldLeft[B](initial: B)(update: (B, A) => B): B
        def map[B](f: A => B): List[B]
        def flatMap[B](f: A => List[B]): List[B]
        def filter(f: A => Boolean): List[A]
        def forall(pred: A => Boolean): Boolean
        def exists(pred: A => Boolean): Boolean
 
        // Common functions
        def length: Int
        def drop(i: Int): List[A]
        def reverse: List[A]
        def flatten[B](/* A debe poder convertirse al tipo List[B] */): List[B]

        // Concatena dos listas
        def ++[B >: A](other: List[B]): List[B]

    object List:
        // Crea una lista con `n` copias del elemento `elem`
        def fill[A](n: Int)(elem: A): List[A] = ???

In [ ]:
def divide_and_conquer[A, B](l: List[A] /*, ... */): B =
    l match
        case Nil =>
            ??? 
        case head :: tail =>
            val tailSol: B = divide_and_conquer(tail)
            ??? /*(head, tailSol)*/

In [ ]:
def iterative[A, B](l: List[A] /* ... */): B =

    def step(out: B, aux: List[A]): B =
        aux match
            case Nil => out
            case head :: tail =>
                step(??? /*(out, head)*/, tail)

    step(???, l)

In [ ]:
// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    
    def void[A]: Tree[A] =
        Empty()
    
    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())
    
    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)
    
    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())
    
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

In [ ]:
import Tree._

In [ ]:
def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() =>
            empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node),
                a,
                foldTree(right)(empty)(node))

# Ejercicio 1
__(4 puntos)__

Se pide implementar la función `unzip` que, dada una lista de pares, devuelve el par de listas formado por los primeros y segundos elementos de cada par, respetando el orden original. Por ejemplo:


In [ ]:
class TestUnzip(unzip: List[(Int, String)] => (List[Int], List[String]))
extends AnyFlatSpec with should.Matchers:
    "unzip" should "work" in:
        unzip(Nil) shouldBe (Nil, Nil)
        unzip(List((1, "a"))) shouldBe (List(1), List("a"))
        unzip(List((1, "a"), (2, "b"), (3, "c"))) shouldBe
            (List(1, 2, 3), List("a", "b", "c"))
        unzip(List((10, "x"), (20, "y"))) shouldBe
            (List(10, 20), List("x", "y"))

__a) (1 punto)__ Implementa la función `unzip` mediante recursión.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def unzip[A, B](l: List[(A, B)]): (List[A], List[B]) = 
    l match 
        case Nil => (Nil, Nil)
        case (a,b) :: tail => 
            val (la, lb) = unzip(tail)
            (a :: la, b :: lb)

In [ ]:
run(TestUnzip(unzip))

__b) (1 punto)__ Implementa la función `unzip` mediante la función de orden superior `foldRight`.

In [ ]:
// IMPLEMENTA TU RESPUESTA
def unzip[A, B](l: List[(A, B)]): (List[A], List[B]) = 
    l.foldRight(Nil, Nil): 
        case ((a,b), (la, lb)) => (a :: la, b :: lb)

In [ ]:
run(TestUnzip(unzip))

__c) (1 punto)__ Implementa la función `unzip` mediante recursión final.

In [ ]:
// IMPLEMENTA TU RESPUESTA
def unzip[A, B](l: List[(A, B)]): (List[A], List[B]) = 
    def step(out: (List[A], List[B]), aux: List[(A, B)]): (List[A], List[B]) = 
        aux match 
            case Nil => out
            case (a, b) :: tail => 
                step((a :: out._1, b :: out._2), tail)
    step((Nil, Nil), l) match 
        case (la, lb) => (la.reverse, lb.reverse)

In [ ]:
run(TestUnzip(unzip))

__d) (1 punto)__ Implementa la función `unzip` mediante la función de orden superior `foldLeft`.

In [ ]:
// IMPLEMENTA TU RESPUESTA
def unzip[A, B](l: List[(A, B)]): (List[A], List[B]) = 
    val (la, lb) = 
        l.foldLeft[(List[A], List[B])](Nil, Nil): 
            case ((la, lb), (a, b)) => (a :: la, b :: lb)
    (la.reverse, lb.reverse)

In [ ]:
run(TestUnzip(unzip))

# Ejercicio 2
__(2 puntos)__

Considera las siguientes dos funciones, ambas completamente implementadas:

- `runningSums` calcula las sumas acumuladas de una lista de enteros, incluyendo el valor inicial 0.
- `runningLengths` calcula las longitudes acumuladas de las cadenas de una lista, incluyendo el valor inicial 0.

Obsérvese que las dos funciones responden al mismo patrón estructural, con las únicas diferencias del valor inicial y la operación de actualización del acumulador.

In [ ]:
def runningSums(l: List[Int]): List[Int] =

    def step(remaining: List[Int], acc: ::[Int]): List[Int] =
        remaining match
            case Nil => acc
            case head :: tail => step(tail, ::(acc.head + head, acc))
            
    step(l, ::(0, Nil)).reverse

In [ ]:
def runningLengths(l: List[String]): List[Int] =

    def step(remaining: List[String], acc: ::[Int]): List[Int] =
        remaining match
            case Nil => acc
            case head :: tail => step(tail, ::(acc.head + head.length, acc))
            
    step(l, ::(0, Nil)).reverse

__a) (1 punto)__ Implementa la función de orden superior `scanLeft` que abstrae el patrón común de las dos funciones anteriores:


In [ ]:
// IMPLEMENTA TU RESPUESTA

def scanLeft[A, B](l: List[A])(initial: B)(update: (B, A) => B): List[B] =

    def step(remaining: List[A], acc: ::[B]): List[B] =
        remaining match
            case Nil => acc
            case head :: tail => step(tail, ::(update(acc.head, head), acc))
            
    step(l, ::(initial, Nil)).reverse

__b) (1 punto)__ Re-implementa `runningSums` y `runningLengths` usando `scanLeft`.

In [ ]:
class TestRunningSums(runningSums: List[Int] => List[Int])
extends AnyFlatSpec with should.Matchers:
    "runningSums" should "work" in:
        runningSums(Nil) shouldBe List(0)
        runningSums(List(1, 2, 3, 4)) shouldBe List(0, 1, 3, 6, 10)
        runningSums(List(5)) shouldBe List(0, 5)

In [ ]:
class TestRunningLengths(runningLengths: List[String] => List[Int])
extends AnyFlatSpec with should.Matchers:
    "runningLengths" should "work" in:
        runningLengths(Nil) shouldBe List(0)
        runningLengths(List("a", "bb", "ccc")) shouldBe List(0, 1, 3, 6)
        runningLengths(List("hola")) shouldBe List(0, 4)

In [ ]:
// IMPLEMENTA TU RESPUESTA

def runningLengths(l: List[String]): List[Int] =
    scanLeft(l)(0)(_ + _.length)

In [ ]:
// IMPLEMENTA TU RESPUESTA

def runningSums(l: List[Int]): List[Int] =
    scanLeft(l)(0)(_ + _)

In [ ]:
run(TestRunningSums(runningSums))
run(TestRunningLengths(runningLengths))

# Ejercicio 3
__(2 puntos)__

La codificación run-length es una técnica de compresión que representa secuencias de elementos iguales consecutivos como un par (elemento, número de repeticiones). Se pide implementar la función de decodificación `decode` que, dada una lista de pares de la forma `(elem, n)`, produce la lista expandida con `n` copias de `elem` por cada par. Por ejemplo:


In [ ]:
class TestDecode(decode: List[(Char, Int)] => List[Char])
extends AnyFlatSpec with should.Matchers:
    "decode" should "work" in:
        decode(Nil) shouldBe Nil
        decode(List(('a', 1))) shouldBe List('a')
        decode(List(('a', 3))) shouldBe List('a', 'a', 'a')
        decode(List(('a', 3), ('b', 2), ('c', 1))) shouldBe
            List('a', 'a', 'a', 'b', 'b', 'c')
        decode(List(('x', 4), ('y', 2))) shouldBe
            List('x', 'x', 'x', 'x', 'y', 'y')

__a) (1 punto)__ Implementa la función `decode` mediante recursión.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def decode[A](l: List[(A, Int)]): List[A] = 
    l match 
        case Nil => Nil
        case (a, n) :: tail => List.fill(n)(a) ++ decode(tail)

In [ ]:
run(TestDecode(decode))

__b) (1 punto)__ Implementa la función `decode` mediante la función de orden superior `flatMap`. Para ello se puede utilizar la función `List.fill` del preámbulo.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def decode[A](l: List[(A, Int)]): List[A] = 
    l.flatMap: (a, n) => 
        List.fill(n)(a)

In [ ]:
run(TestDecode(decode))

# Ejercicio 4
__(2 puntos)__

Se pide implementar la función `mirror` que, dado un árbol binario, produce su imagen especular: el árbol resultante de intercambiar recursivamente el subárbol izquierdo y el derecho en cada nodo. Por ejemplo:


In [ ]:
class TestMirror(mirror: Tree[Int] => Tree[Int])
extends AnyFlatSpec with should.Matchers:
    "mirror" should "work" in:
        mirror(void) shouldBe void
        mirror(leaf(1)) shouldBe leaf(1)
        mirror(left(leaf(1), 2)) shouldBe right(2, leaf(1))
        mirror(right(3, leaf(1))) shouldBe left(leaf(1), 3)
        mirror(node(leaf(1), 2, leaf(3))) shouldBe node(leaf(3), 2, leaf(1))
        mirror(node(leaf(1), 2, node(leaf(3), 4, leaf(5)))) shouldBe
            node(node(leaf(5), 4, leaf(3)), 2, leaf(1))

__a) (1 punto)__ Implementa la función `mirror` mediante recursión.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def mirror[A](t: Tree[A]): Tree[A] = 
    t match 
        case Empty() => Empty()
        case Node(left, root, right) => Node(mirror(right), root, mirror(left))

In [ ]:
run(TestMirror(mirror))

__b) (1 punto)__ Implementa la función `mirror` mediante la función `foldTree`.

In [ ]:
// IMPLEMENTA TU RESPUESTA

def mirror[A](t: Tree[A]): Tree[A] = 
    foldTree(t)(Empty()): (mirrorLeft, root, mirrorRight) => 
        Node(mirrorRight, root, mirrorLeft)

In [ ]:
run(TestMirror(mirror))